In [1]:
# =======================================================
# NOTEBOOK 4: O TESTE DE SANIDADE (REGRESSÃO LINEAR MÚLTIPLA)
# =======================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression # <--- O NOVO MOTOR AQUI
import warnings
warnings.filterwarnings('ignore')

# -------------------------------------------------------
# 1. FUNÇÕES DO JUIZ (O SCORE OFICIAL)
# -------------------------------------------------------
def norm_f(m, a=4443.76, b=1.53, c=4443.76, d=0):
    return (a / ((m**b) + c)) + d

def calcular_score(df_resultados, alpha_peso=2.0):
    resultados_viagem = []

    for trem_id, df_trem in df_resultados.groupby('ID_Trem'):
        T = len(df_trem)
        if T == 0: continue
        
        df_trem = df_trem.sort_values('Ciclo_Atual')
        erro_t = df_trem['RUL_Real'] - df_trem['RUL_Final'] 
        
        # SDE e RMSE
        erro_medio = erro_t.mean() 
        diferencas_quadradas = (erro_t - erro_medio) ** 2 
        metric_val = np.sqrt(np.sum(diferencas_quadradas) / T)
        rmse_val = np.sqrt(np.sum(erro_t ** 2) / T)

        # NORMALIZAÇÃO
        sde_norm = norm_f(metric_val)
        rmse_norm = norm_f(rmse_val)
        
        # ACCURACY
        erro_absoluto = np.abs(df_trem['RUL_Real'] - df_trem['RUL_Final']) 
        rul_real = np.maximum(df_trem['RUL_Real'], 1e-5) 
        valores_exponenciais = np.exp(-(erro_absoluto / rul_real)) 
        acc_val = valores_exponenciais.mean() 

        # PROGNOSTIC HORIZON (PH)
        t_eof = df_trem['Ciclo_Atual'].max() 
        margem_constante = t_eof * 0.10
        limite_inferior = df_trem['RUL_Real'] - margem_constante
        limite_superior = df_trem['RUL_Real'] + margem_constante
        
        dentro_limites = (df_trem['RUL_Final'] >= limite_inferior) & (df_trem['RUL_Final'] <= limite_superior)
        fora_limites = ~dentro_limites 
        
        if fora_limites.any():
            ultimo_erro = df_trem.loc[fora_limites, 'Ciclo_Atual'].max()
            df_reta_final = df_trem[df_trem['Ciclo_Atual'] > ultimo_erro]
            t_alpha = t_eof if df_reta_final.empty else df_reta_final['Ciclo_Atual'].min()
        else:
            t_alpha = df_trem['Ciclo_Atual'].min()
                
        ph = (t_eof - t_alpha) / t_eof if t_eof > 0 else 0
        score = (rmse_norm + sde_norm + (alpha_peso * ph)) / (2 + alpha_peso)

        resultados_viagem.append({
            'ID_Trem': trem_id, 'T_Pontos': T, 'Erro_Medio': round(erro_medio, 2),
            'RMSE_Bruto': round(rmse_val, 2), 'Accuracy': round(acc_val, 4),
            'PH': round(ph, 4), 'Score': round(score, 4)
        })

    df_metricas = pd.DataFrame(resultados_viagem)
    print("\n--- Métricas Individuais por Trem (Regressão Linear) ---")
    print(df_metricas.to_string(index=False))
    print("\n" + "="*70)
    print(f" 🎯 SCORE MÉDIO FINAL: {df_metricas['Score'].mean():.4f} | PH Médio: {df_metricas['PH'].mean():.4f}")
    print("="*70)

# -------------------------------------------------------
# 2. CARREGAMENTO DOS DADOS V3
# -------------------------------------------------------
print("1. Carregando o Dataset V3...")
df_ml = pd.read_csv('dataset_features_v3.csv')

df_treino = df_ml[df_ml['Grupo'] == 'Treino'].copy()
df_val = df_ml[df_ml['Grupo'] == 'Validação'].copy()

features = [
    'Distancia_Ref_Atual', 'Inclinacao_Mecanica', 
    'Vibracao_Max', 'Vibracao_Media', 'Vibracao_Tendencia', 
    'Vibracao_Std', 'Vibracao_Kurtosis', 'Vibracao_Energia'
]

X_train = df_treino[features]
y_train_bruto = df_treino['RUL_Alvo'].values
X_test = df_val[features]
y_test_bruto = df_val['RUL_Alvo'].values

rul_maximo = 400
y_train = np.clip(y_train_bruto, a_min=0, a_max=rul_maximo)
y_test = np.clip(y_test_bruto, a_min=0, a_max=rul_maximo)

# -------------------------------------------------------
# 3. TREINAMENTO: A REGRESSÃO LINEAR
# -------------------------------------------------------
print("2. Treinando a Regressão Linear Múltipla...")
modelo_lr = LinearRegression(n_jobs=-1)
modelo_lr.fit(X_train, y_train)

# -------------------------------------------------------
# 4. FILTROS E PREVISÃO
# -------------------------------------------------------
print("3. Prevendo e aplicando Filtros Industriais...")
previsoes_brutas = modelo_lr.predict(X_test)

previsoes_brutas = np.where(previsoes_brutas > 360, 400, previsoes_brutas)

df_resultados = pd.DataFrame({
    'ID_Trem': df_val['ID_Trem'],
    'Ciclo_Atual': df_val['Ciclo'],
    'RUL_Real': y_test,               
    'RUL_Real_Bruto': y_test_bruto,   
    'RUL_Previsto_Bruto': previsoes_brutas,
})

df_resultados = df_resultados.sort_values(['ID_Trem', 'Ciclo_Atual'])

df_resultados['RUL_Suavizado'] = df_resultados.groupby('ID_Trem')['RUL_Previsto_Bruto'].transform(
    lambda x: x.ewm(span=3, adjust=False).mean()
)
df_resultados['RUL_Final'] = df_resultados.groupby('ID_Trem')['RUL_Suavizado'].cummin()

calcular_score(df_resultados)

# -------------------------------------------------------
# 5. O PAINEL DE RAIO-X (GRÁFICOS)
# -------------------------------------------------------
print("\n4. Gerando o Painel de Avaliação da Frota (Gráficos)...")
trens_validacao = df_resultados['ID_Trem'].unique()
num_trens = len(trens_validacao)
cols = 3
rows = int(np.ceil(num_trens / cols))
fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
axes = axes.flatten()

for idx, trem in enumerate(trens_validacao):
    df_trem = df_resultados[df_resultados['ID_Trem'] == trem].sort_values(by='Ciclo_Atual')
    
    ciclos = df_trem['Ciclo_Atual'].values
    y_real = df_trem['RUL_Real'].values
    y_pred = df_trem['RUL_Final'].values
    
    t_eof = ciclos.max() 
    margem_constante = t_eof * 0.10
    limite_inferior = y_real - margem_constante
    limite_superior = y_real + margem_constante
    
    dentro_limites = (y_pred >= limite_inferior) & (y_pred <= limite_superior)
    fora_limites = ~dentro_limites 
    
    if fora_limites.any():
        ultimo_erro = df_trem.loc[fora_limites, 'Ciclo_Atual'].max()
        df_reta_final = df_trem[df_trem['Ciclo_Atual'] > ultimo_erro]
        t_alpha = t_eof if df_reta_final.empty else df_reta_final['Ciclo_Atual'].min()
    else:
        t_alpha = df_trem['Ciclo_Atual'].min()
            
    ph = (t_eof - t_alpha) / t_eof if t_eof > 0 else 0
    
    ax = axes[idx]
    ax.fill_between(ciclos, limite_inferior, limite_superior, color='gray', alpha=0.15, label='Margem de Tolerância (±10%)')
    ax.plot(ciclos, y_real, color='black', linewidth=2, label='RUL Real (Cap 400)')
    
    cor_linha = '#2980b9' if ph > 0.8 else ('#27ae60' if ph > 0.4 else '#e74c3c')
    ax.plot(ciclos, y_pred, color=cor_linha, linewidth=2.5, linestyle='-', label='Regressão Linear + EMA')
    
    ax.set_title(f'Train_{trem:02d} | PH: {ph:.4f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Ciclos de Vida (Viagens)', fontsize=10)
    ax.set_ylabel('RUL Restante', fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_ylim(-20, 420)
    
    if idx == 0: ax.legend(fontsize=9)

for i in range(num_trens, len(axes)): fig.delaxes(axes[i])
plt.tight_layout()
plt.show()

1. Carregando o Dataset V3...


FileNotFoundError: [Errno 2] No such file or directory: 'dataset_features_v3.csv'